# Step 2: Classify Texts

This notebook performs zero-shot classification of all proposal titles into the merged clusters identified in Step 1.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "llama-3.3-70b-instruct"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
DATA_OUT_BASE_PATH = "../out/"
chunk_clusters_fp = f"{DATA_OUT_BASE_PATH}2.chunk_clusters.json"
merged_clusters_fp = f"{DATA_OUT_BASE_PATH}2.merged_clusters.json"
classified_proposals_fp = f"{DATA_OUT_BASE_PATH}3.classified_proposals.json"
classified_proposals_errors_fp = f"{DATA_OUT_BASE_PATH}3.classified_proposals_errors.json"
clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"

In [ ]:
clusters = json.load(open(merged_clusters_fp, 'r'))
cluster_names = [c['name'] for c in clusters]
list_proposals = json.load(open("../data/proposals.json"))
list_titles = [x["title"] for x in list_proposals]
list_texts = [x["text"] for x in list_proposals]

In [ ]:
prompt_classify = """Your task is to find the most appropriate class for a given text.
In what follows you can find the set of classes together with their definition.
You can choose one of the following classes:
%s
Output a JSON with the following structure, do not output anything else.
Before outputin check the JSON is valid. Correct it if it is not

Output:
    class: str # the class corresponding to the given text.
"""
for cluster in clusters:
    prompt_classify %= (f"- {cluster['name']}: {cluster['summary']}\n%s")
prompt_classify %= ""
prompt_classify

In [ ]:
# Test prompt
r = model_json(prompt_classify, list_titles[0])
print(r)

In [ ]:
s = set()
for proposal in list_proposals:
    s.add(proposal['guid'])
len(s)

In [ ]:
classification_results = []
errors = []
for idx, title in enumerate(tqdm(list_titles)):
    try:
        r = model_json(prompt_classify, title)
        proposal = list_proposals[idx]
        # assert r['class'] in cluster_names
        if r['class'] in cluster_names:
            classification_results.append({
                'title': title,
                'cluster': r['class'],
                'language': proposal['language'],
                'UGC': proposal.get('topictitle', None),
                'id': proposal['guid']
            })
        else:
            print("Error")
            errors.append({
                'title': title,
                'id': proposal['guid']
            })
    except:
        print("Error")
        errors.append({
            'title': title,
            'id': proposal['guid'],
            'non_clustering_error': True
            })
    if idx % 500 == 0:
        json.dump(classification_results, open(classified_proposals_fp, 'w'), indent=2, ensure_ascii=False)
        json.dump(errors, open(classified_proposals_errors_fp, 'w'), indent=2, ensure_ascii=False)
json.dump(classification_results, open(classified_proposals_fp, 'w'), indent=2, ensure_ascii=False)
json.dump(errors, open(classified_proposals_errors_fp, 'w'), indent=2, ensure_ascii=False)

clusterized_proposals = defaultdict(list)

for proposal in classification_results:
    clusterized_proposals[proposal['cluster']].append({
        'id': proposal['id'],
        'title': proposal['title'],
        'language': proposal['language'],
        'UGC': proposal['UGC'],
    })
    
json.dump(clusterized_proposals, open(clusterized_proposals_fp, 'w'), indent=2, ensure_ascii=False)

In [ ]:
classification_results = json.load(open(classified_proposals_fp))
clusterized_proposals = defaultdict(list)

for proposal in classification_results:
    clusterized_proposals[proposal['cluster']].append({
        'id': proposal['id'],
        'title': proposal['title'],
        'language': proposal['language'],
        'UGC': proposal['UGC'],
    })

json.dump(clusterized_proposals, open(clusterized_proposals_fp, 'w'), indent=2, ensure_ascii=False)